In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re, torch
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'
out_dir = f'{data_dir}/save_hiddens'

model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

in_file_path = f'{data_dir}/msnap_gpt-5.2_checked_contexts_{model_name}.json'
datas = json_utils.load_json(in_file_path)

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

In [ ]:
print(json_utils.to_str(datas[0]))

In [ ]:
def make_prompts(datas):
    prompts_fact, prompts_counter = [], []
    answers_fact, answers_counter = [], []
    cnt_fact, cnt_counter = 0, 0

    for data in datas:
        query = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        context_fact_dict: dict = data['contexts_fact']
        context_counter_dict: dict = data['contexts_counter']

        contexts_fact = list(context_fact_dict.values())
        contexts_counter = list(context_counter_dict.values())

        cnt_fact += len(contexts_fact)
        cnt_counter += len(contexts_counter)

        for context_fact in contexts_fact:
            prompts_fact.append(msnap_prompts.get_generate_prompt(query, [context_fact]))
            answers_fact.append(answer_fact)
        
        for context_counter in contexts_counter:
            prompts_counter.append(msnap_prompts.get_generate_prompt(query, [context_counter]))
            answers_counter.append(answer_counter)
    
    print(f'make_prompts() cnt_fact : {cnt_fact}')
    print(f'make_prompts() cnt_counter : {cnt_counter}\n')
    print(f'make_prompts() prompts_fact size : {len(prompts_fact)}')
    print(f'make_prompts() answers_fact size : {len(answers_fact)}\n')
    print(f'make_prompts() prompts_counter size : {len(prompts_counter)}')
    print(f'make_prompts() answers_counter size : {len(answers_counter)}\n')

    return prompts_fact, prompts_counter, answers_fact, answers_counter

In [ ]:
prompts_fact, prompts_counter, answers_fact, answers_counter = make_prompts(datas)

In [ ]:
def save_hiddens(prompts, group_name, batch_size, out_prefix):
    print(f'group_name : {group_name}, prompts size : {len(prompts)}')

    num_layers = model.config.num_hidden_layers
    print(f'save_hiddens() num_layers : {num_layers}\n')

    # outputs.hidden_states는 (초기 임베딩 + 각 레이어 출력)이므로 길이는 num_layers + 1
    h_states_accumulator = {i: [] for i in range(num_layers+1)}

    for prompts_batch in chunks(prompts, batch_size):
        outputs = model_utils.forward(
            model, tokenizer, device,
            prompts_batch, max_seq_length
        )

        for layer_idx, h_states in enumerate(outputs.hidden_states):
            # 마지막 토큰 추출 -> CPU로 이동 -> float32 캐스팅
            last_token_h_state = h_states[:, -1, :].cpu().to(torch.float32)
            h_states_accumulator[layer_idx].append(last_token_h_state)

    # 리스트에 모인 배치 텐서들을 하나의 큰 텐서로 병합 후 파일로 저장
    for layer_idx in range(num_layers+1):
        layer_h_states = torch.cat(h_states_accumulator[layer_idx], dim=0)
        print(f'layer_{layer_idx}_h_states shape : {layer_h_states.shape}')

        out_file_path = f'{out_prefix}/{group_name}/layer_{layer_idx}_h_states.pt'
        file_utils.make_parent(out_file_path)
        torch.save(layer_h_states, out_file_path)
        # print(f'layer_{layer_idx} saved -> {out_file_path}\n')
    print()

In [ ]:
out_prefix = f'{out_dir}/{model_name}'
save_hiddens(prompts_fact, 'fact', 20, out_prefix)
save_hiddens(prompts_counter, 'counter', 20, out_prefix)